In [ ]:
# فهم سبب عدم إرسال البيانات

هذا الدفتر يساعد على تتبع سبب عدم إرسال الطلبات إلى Google Sheets. سنقوم بإنشاء بيانات اختبارية، وإرسالها إلى نقطة الويب (Google Apps Script webhook)، ثم قراءة الاستجابة.
</VSCode.Cell>
<VSCode.Cell language="python">
# تحقق من عنوان الـ webhook وبعض بيانات الاختبار
webhook_url = "https://script.google.com/macros/s/AKfycbz_51KMGpbSfJOagqy0GA_VsZct3gqFBGlZszdKzTXheDYeOeR89m4AmHKK633ssPOU/exec"
print("Webhook URL:", webhook_url)

sample_order = {
    "id": "debug-order-1",
    "customerName": "Test User",
    "customerPhone": "0000000000",
    "wilaya": "Test Wilaya",
    "commune": "Test Commune",
    "address": "Test Address",
    "deliveryType": "home",
    "paymentMethod": "cod",
    "deliveryCompany": "Test Delivery",
    "deliveryPrice": 0,
    "subtotal": 100,
    "couponCode": "",
    "couponDiscount": 0,
    "total": 100,
    "status": "pending",
    "source": "website",
    "createdAt": "2026-05-05T00:00:00Z",
    "items": [
        {
            "productId": "prod-1",
            "name": "Test Product",
            "quantity": 1,
            "size": "M",
            "color": "Red",
            "unitPrice": 100,
            "totalPrice": 100,
        }
    ],
}

print("Sample order payload built successfully.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## إعداد الاعتمادات وواجهة Google Sheets API

في هذه الحالة، نحن نستخدم Google Apps Script Web App كخادم وسيط. إذا أردنا استخدام Google Sheets API مباشرة، نحتاج إلى إعداد بيانات الاعتماد و `gspread` أو `googleapiclient`.

لكن أولاً يجب التأكد من أن نقطة الويب تعمل بشكل صحيح قبل الانتقال إلى تكامل API مباشر.
</VSCode.Cell>
<VSCode.Cell language="python">
import json
import urllib.request

print("Sending test payload to webhook...")

req = urllib.request.Request(
    webhook_url,
    data=json.dumps(sample_order).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)

try:
    with urllib.request.urlopen(req, timeout=20) as response:
        status = response.getcode()
        body = response.read().decode("utf-8")
        headers = dict(response.getheaders())
        print("STATUS:", status)
        print("HEADERS:", headers)
        print("BODY:", body)
except Exception as error:
    print("ERROR:", repr(error))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## إنشاء بيانات الاختبار

نحن استخدمنا الكائن `sample_order` أعلاه. إذا كانت تفاصيل الطلب الحقيقية مختلفة، فتأكد من أن خاصية `items` مصاغة كما في الكود الموجود في `src/lib/googleSheets.ts`.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## التحقق من الاستجابة والأخطاء

إذا كان الاستدعاء ناجحًا، يجب أن ترى `STATUS 200` واستجابة JSON تحتوي على `{ "success": true }`.

إذا حدث خطأ، راقب رسالة الخطأ الكاملة لمعرفة ما إذا كانت المشكلة:
- endpoint غير صالح
- خطأ وقت التشغيل في Apps Script
- خطأ في JSON
- مشكلة في الشبكة أو CORS
</VSCode.Cell>
<VSCode.Cell language="markdown">
## استخدام السجلات لتتبع المشكلة

يمكنك أيضًا إضافة طباعة في `src/lib/googleSheets.ts` داخل `sendOrderToGoogleSheets` لمراقبة:
- `webhookUrl`
- `payload`
- `response.status`
- `responseText`

هذا يساعد في معرفة إن كان الطلب يُرسل بالفعل أم أن المشكلة تحدث قبل ذلك.
